# Address Geocoding
This notebook geocodes all unique HDB block addresses in the dataset to latitude and longitude coordinates via the OneMap API. The resulting coordinates are used in the next notebook to calculate distances to amenities such as MRT stations, schools, malls, and hawker centres.

## Setup
Import required libraries, load the OneMap API token from the environment and extract all unique block and street name combinations from the raw dataset.

We geocode unique addresses only and not all 233,055 transactions since many transactions share the same block. This reduces the number of API calls from 233,055 to 9,712.

In [55]:
import pandas as pd
import numpy as np
import requests
import time
import os
from dotenv import load_dotenv

load_dotenv('../.env')
token = os.getenv('ONEMAP_TOKEN')

# Verify token loaded
print("Token loaded:", token is not None)

df = pd.read_csv('../data/raw/hdb_resale_transactions.csv')
unique_addresses = df[['block', 'street_name']].drop_duplicates().reset_index(drop=True)

print(f"Total transactions: {len(df):,}")
print(f"Unique addresses:   {len(unique_addresses):,}")

Token loaded: True
Total transactions: 233,055
Unique addresses:   9,712


## Geocoding Function
The `geocode_address` function queries the OneMap search API with a block number and street name, returning latitude and longitude coordinates.

Key implementation details:
- Requires an authenticated token in the `Authorization` header — unauthenticated requests are rate limited to far fewer calls per minute
- Retries up to 3 times on failure with a 1 second delay between attempts
- Returns `None, None` if all retries are exhausted

In [ ]:
def geocode_address(block, street_name, token, retries=3):
    query = f"{block} {street_name} SINGAPORE"
    url = "https://www.onemap.gov.sg/api/common/elastic/search"
    params = {
        "searchVal": query,
        "returnGeom": "Y",
        "getAddrDetails": "Y",
        "pageNum": 1
    }
    headers = {
        "Authorization": token
    }
    for attempt in range(retries):
        try:
            response = requests.get(url, params=params, headers=headers, timeout=10)
            data = response.json()
            if data.get('found', 0) > 0:
                result = data['results'][0]
                return float(result['LATITUDE']), float(result['LONGITUDE'])
            else:
                time.sleep(1)
        except Exception as e:
            time.sleep(1)
    return None, None

# Test on one address
block, street = unique_addresses.iloc[0]['block'], unique_addresses.iloc[0]['street_name']
lat, lon = geocode_address(block, street, token)
print(f"Test address: {block} {street}")
print(f"Coordinates:  {lat}, {lon}")

## Validation Test
Before running the full geocoding loop, we validate the function on a single address and then on a 50 address subset to confirm the API is responding correctly and the token is working.

In [19]:
test = unique_addresses.head(50).copy().reset_index(drop=True)
results = []

for idx, row in test.iterrows():
    lat, lon = geocode_address(row['block'], row['street_name'], token)
    results.append({
        'block': row['block'],
        'street_name': row['street_name'],
        'latitude': lat,
        'longitude': lon
    })
    time.sleep(0.25)

test_results = pd.DataFrame(results)
print(f"Success: {test_results['latitude'].notna().sum()} / 50")
print(f"Failed:  {test_results['latitude'].isna().sum()} / 50")

Success: 50 / 50
Failed:  0 / 50


## Full Geocoding Loop
We geocode all 9,712 unique addresses with a 0.25 second delay between requests to stay within the OneMap rate limit of 300 calls per minute. Results are saved to CSV every 500 addresses so progress is not lost if the kernel resets.

In [22]:
results = []

for idx, row in unique_addresses.iterrows():
    lat, lon = geocode_address(row['block'], row['street_name'], token)
    results.append({
        'block': row['block'],
        'street_name': row['street_name'],
        'latitude': lat,
        'longitude': lon
    })
    
    if (idx + 1) % 500 == 0:
        print(f"Processed {idx + 1:,} / {len(unique_addresses):,}")
        pd.DataFrame(results).to_csv('../data/processed/geocoded_addresses.csv', index=False)
    
    time.sleep(0.25)

geocoded = pd.DataFrame(results)
geocoded.to_csv('../data/processed/geocoded_addresses.csv', index=False)
print(f"Done. Geocoded {len(geocoded):,} addresses.")

Processed 500 / 9,712
Processed 1,000 / 9,712
Processed 1,500 / 9,712
Processed 2,000 / 9,712
Processed 2,500 / 9,712
Processed 3,000 / 9,712
Processed 3,500 / 9,712
Processed 4,000 / 9,712
Processed 4,500 / 9,712
Processed 5,000 / 9,712
Processed 5,500 / 9,712
Processed 6,000 / 9,712
Processed 6,500 / 9,712
Processed 7,000 / 9,712
Processed 7,500 / 9,712
Processed 8,000 / 9,712
Processed 8,500 / 9,712
Processed 9,000 / 9,712
Processed 9,500 / 9,712
Done. Geocoded 9,712 addresses.


## Failure Analysis
We inspect addresses that failed to geocode and assess how many transactions are affected. 95 addresses failed on the initial run, affecting 2,353 transactions (1.0% of the dataset).

Failures are caused by abbreviated street names that OneMap does not recognise — for example `C'WEALTH` instead of `COMMONWEALTH` and `PK` instead of `PARK`.

In [24]:
print(f"Failed: {geocoded['latitude'].isna().sum()}")


Failed: 95


In [25]:
failed = geocoded[geocoded['latitude'].isna()]
print(f"Failed: {len(failed)}")
print(failed.head(10).to_string())

Failed: 95
    block        street_name  latitude  longitude
269    32         NEW MKT RD       NaN        NaN
335   411  C'WEALTH AVE WEST       NaN        NaN
568     3    ST. GEORGE'S RD       NaN        NaN
583    12       FARRER PK RD       NaN        NaN
592    21    ST. GEORGE'S RD       NaN        NaN
686    83        C'WEALTH CL       NaN        NaN
689    81        C'WEALTH CL       NaN        NaN
691   109      C'WEALTH CRES       NaN        NaN
692    94        C'WEALTH DR       NaN        NaN
693   111      C'WEALTH CRES       NaN        NaN


In [26]:
failed_addresses = geocoded[geocoded['latitude'].isna()][['block', 'street_name']]
affected = df.merge(failed_addresses, on=['block', 'street_name'])
print(f"Transactions affected: {len(affected):,}")
print(f"As % of dataset: {len(affected)/len(df)*100:.1f}%")

Transactions affected: 2,353
As % of dataset: 1.0%


## Abbreviation Expansion
We define a dictionary of common abbreviations and a regex-based expansion function to standardise street names before querying the API.

Regex word boundaries (`\b`) are used instead of simple string replacement to prevent partial matches. For example, ensuring `DR` expands to `DRIVE` only as a standalone word and not inside words like `DRIVE` itself.

In [54]:
import re

abbreviations = {
    r"\bC'WEALTH\b": "COMMONWEALTH",
    r"\bST\.": "SAINT",
    r"\bAVE\b": "AVENUE",
    r"\bRD\b": "ROAD",
    r"\bCL\b": "CLOSE",
    r"\bCRES\b": "CRESCENT",
    r"\bDR\b": "DRIVE",
    r"\bST\b": "STREET",
    r"\bBT\b": "BUKIT",
    r"\bUPP\b": "UPPER",
    r"\bKG\b": "KAMPONG",
    r"\bPK\b": "PARK",
    r"\bMKT\b": "MARKET",
}

def expand_abbreviations(street_name):
    for pattern, full in abbreviations.items():
        street_name = re.sub(pattern, full, street_name)
    return street_name

# Test
test_cases = ["SPOTTISWOODE PK RD", "EVERTON PK", "BIDADARI PK DR", "C'WEALTH AVE WEST", "ST. GEORGE'S RD"]
for t in test_cases:
    print(f"{t} -> {expand_abbreviations(t)}")

SPOTTISWOODE PK RD -> SPOTTISWOODE PARK ROAD
EVERTON PK -> EVERTON PARK
BIDADARI PK DR -> BIDADARI PARK DRIVE
C'WEALTH AVE WEST -> COMMONWEALTH AVENUE WEST
ST. GEORGE'S RD -> SAINT GEORGE'S ROAD


## Retry Failed Addresses
We retry all 95 failed addresses with expanded street names. All 95 geocode successfully after abbreviation expansion.

The expanded geocoded coordinates are merged back into the main geocoded DataFrame and the final result is saved to `data/processed/geocoded_addresses.csv`.

Final result: 9,712 addresses geocoded, 0 failures.

In [46]:
retry_results = []

for idx, row in failed_addresses.iterrows():
    expanded_street = expand_abbreviations(row['street_name'])
    lat, lon = geocode_address(row['block'], expanded_street, token)
    retry_results.append({
        'block': row['block'],
        'street_name': row['street_name'],
        'latitude': lat,
        'longitude': lon
    })
    time.sleep(0.25)

retry_df = pd.DataFrame(retry_results)
print(f"Retry success: {retry_df['latitude'].notna().sum()} / {len(retry_df)}")
print(f"Still failed: {retry_df['latitude'].isna().sum()}")

Retry success: 95 / 95
Still failed: 0


In [52]:
for _, row in retry_df[retry_df['latitude'].notna()].iterrows():
    mask = (geocoded['block'] == row['block']) & (geocoded['street_name'] == row['street_name'])
    geocoded.loc[mask, 'latitude'] = row['latitude']
    geocoded.loc[mask, 'longitude'] = row['longitude']

geocoded.to_csv('../data/processed/geocoded_addresses.csv', index=False)
print(f"Final geocoded addresses: {len(geocoded):,}")
print(f"Successful: {geocoded['latitude'].notna().sum():,}")
print(f"Failed: {geocoded['latitude'].isna().sum():,}")

Final geocoded addresses: 9,712
Successful: 9,712
Failed: 0
